# Fine-tune the prompt-attack classifier on a Colab GPU

Runtime → Change runtime type → **T4 GPU**. This runs the same `classifier/train.py` as the local CPU runs, so metrics land in `results/experiments.jsonl` in the same format.

The decision threshold is chosen on the validation split, never the test split.

In [ ]:
!git clone -b finetuned-classifier https://github.com/aadraxe/LLM-middleware-security.git
%cd LLM-middleware-security
!pip install -q -r requirements.txt -r requirements-classifier.txt

## 1. Build the dataset
Downloads five Hugging Face datasets (licenses in `data/DATASET.md`) and writes seeded splits. Check the printed `sha256` values against `data/processed/summary.json` in the repo to confirm you got the same splits.

In [ ]:
!python data/build_dataset.py | tail -20

## 2. Train
On a T4 GPU each run takes a few minutes. You can raise `--epochs` or `--max-len` here; CPU runs used the defaults.

In [ ]:
!python classifier/train.py --model distilbert-base-uncased --run-name distilbert-colab --epochs 3
!python classifier/train.py --model microsoft/deberta-v3-small --run-name deberta-small-colab --epochs 3
!python classifier/train.py --model microsoft/deberta-v3-base --lora --run-name deberta-base-lora-colab --epochs 3

## 3. Compare runs

In [ ]:
import json
for line in open('results/experiments.jsonl'):
    r = json.loads(line)
    print(f"{r['run_name']:28} thr={r['threshold']:.2f} test_f1={r['test']['f1']:.3f} bench_f1={r['benchmark_200']['f1']:.3f} bench_recall={r['benchmark_200']['recall']:.3f} bench_fpr={r['benchmark_200']['fpr']:.3f}")

## 4. Download the model and metrics
Unzip into the repo's `models/` folder locally, append the new lines from `experiments.jsonl` to your copy, then run `python benchmark/compare_classifiers.py --model-dir models/<run>`.

In [ ]:
!zip -qr models.zip models results
from google.colab import files
files.download('models.zip')